# Quantra — Barbershop Fast Loop (Colab)

**Rulebook (for the Risk Doctor / any LLM):** `docs/MLP_INTERPRETABILITY_LAYER.md`.
**Master manual:** `docs/PROJECT_GUIDE.md` — §4.10 (this loop), §4.11 (Policy Registry),
§4.12 (Runtime Override System), §4.13 (Repo Safety), §4.14 (Colab GPU Setup).

**What this is.** The *Barbershop* — *"get the haircut before going to school."* You pick a
small window of FTMO challenge days, run the policy fast, watch what breaks (which days fail
and why), make **one** educated edit via the `OVERRIDES` dict, and repeat. The sole mission of
the policy is to **pass FTMO-style challenges**: hit **+2.5%/day** without breaching a
**−4% trailing wall** on real MT5 bars.

**Two modes, entered freely (not sequential):**
- **Barbershop mode** (this notebook) — fast, operator-driven diagnose-and-shape.
- **Full Training mode** (`colab/Quantra_Train.ipynb`) — long walk-forward runs that
  *generalize* the shaped policy across regimes.

**Repo safety.**
- **ACTIVE WORK** (edits/commits here): `github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-`
- **FALLBACK / SAFE RESTORE** (never edited — emergency revert only): `github.com/monty313/final-rl-model-6_13`

**Known gaps (always honest):** (1) **no real trained model yet** — the #1 active item now that
the gate lockout is fixed (the 3 gates became phase-gated OBSERVATION signals, 2026-06-18, so
`PHASE_FREE` trades freely — but that's unproven on real bars until a real run); (2) the sim
models ONE wall, real FTMO has TWO (the −10% max is an observation via C12, not enforced);
(3) Screen 1 is a labelled demo curve until a real pass-rate series is logged; (4) trade-autopsy
attribution is input×gradient, not SHAP.

> ⚠️ **This notebook is a SKELETON.** The loop plumbing (live output, checkpoint-on-interrupt,
> Policy Registry write with auto-naming, ngrok dashboard) is real and runnable. The single
> integration point — `barbershop_run_day()` in Cell 5 — ships as a clearly-labelled stub
> (`DEMO_MODE`) producing **placeholder** metrics until it is wired to `TradingEnv`.

## Cell 1 — Mount Drive · clone the ACTIVE repo · install deps · race the hardware

In [ ]:
# ── Cell 1 ─ Startup (run once per Colab session; guarded so re-runs are instant) ──
# FTMO framing: every second of CPU setup is a second not spent learning to pass the
# challenge, so the slow steps here are guarded to run ONCE per session.
import os, sys, subprocess, time

REPO_DIR = "quantra_repo"
# ACTIVE WORK repo (we edit/commit here). FALLBACK = final-rl-model-6_13 (NEVER edited) —
# see PROJECT_GUIDE.md §4.13 Repo Safety Protocol.
ACTIVE_REPO = "https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", ACTIVE_REPO, REPO_DIR], check=True)
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Mount Drive once (price data + checkpoints persist across Colab disconnects).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped (not on Colab?):", e)

# Install only the extras Colab doesn't ship. Sentinel guard → instant on re-run.
_SENTINEL = "/content/.quantra_deps_installed"
if not os.path.exists(_SENTINEL):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gdown", "pyarrow", "psutil", "optuna", "seaborn",
                    "statsmodels", "gymnasium", "tqdm", "pyngrok"], check=True)
    import torch
    if torch.cuda.is_available():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"], check=True)
    open(_SENTINEL, "w").close()

# §4.14 PERFORMANCE RULE: call the hardware auto-optimizer at STARTUP so training uses
# ~80% of whichever device is genuinely faster → more passes validated per GPU-hour.
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs
ensure_dirs()
RT = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as _mon:
    HW = plan(state_dim=RT.nominal_state_dim, hw=RT.hardware)
print_report(HW)
print("Device:", HW.device_label, "| n_envs:", HW.n_envs)

## Cell 2 — Build the data cache ONCE (CSV → features → memmap), reuse forever

In [ ]:
# ── Cell 2 ─ Cache-once data pipeline ──
# FTMO framing: the policy can only learn to pass on bars it can see FAST. Parse +
# feature-build is the slow CPU step; do it once, cache it, keep every pass GPU-bound.
from quantra.market_pipeline.data_loader import load_symbol
from quantra.env.trading_env import prepare_symbol_data

SYMBOL = "EURUSD"   # the binding case for real-bar training (#1 gap — see PROJECT_GUIDE §7)

# path=None → loader resolves bars itself: Parquet cache → Drive mount → gdown by ID.
df, rep = load_symbol(SYMBOL, path=None)
print(f"[load] {len(df):,} bars  {df.index.min()} -> {df.index.max()}  source={rep.source}")

# prepare_symbol_data builds the locked 207-feature observation (memmap-cached by the
# feature builder). Re-running is cheap once the cache exists.
SD = prepare_symbol_data(df, symbol=SYMBOL)
print(f"[features] T={len(SD.close):,}  warmup={SD.valid_from:,}  usable={len(SD.close)-SD.valid_from:,}")

## Cell 3 — OPERATOR EDITS HERE: the window + the OVERRIDES

In [ ]:
# ── Cell 3 ─ INPUTS + OVERRIDES (the only cell you normally edit) ──

# ---- INPUTS ------------------------------------------------------------------
POLICY_NAME         = "v1-baseline"    # operator label; the FINAL saved name is AUTO-GENERATED
                                       # from the OVERRIDES diff (Cell 6), never hand-typed.
START_DATE          = "2023-03-01"     # where in the data to start the challenge window
N_DAYS              = 8                 # consecutive FTMO challenge days to train on
N_PASSES            = 20                # how many times the loop repeats over those N_DAYS
CHECKPOINT_INTERVAL = 5                 # save weights + telemetry + registry every N passes
RESUME_FROM         = None             # path to a checkpoint to continue from, or None = fresh

# ---- OVERRIDES (injected at launch — NO edits to config.py / laws.py needed) --
# Every knob is a runtime override saved to the Policy Registry, so you always know which
# configuration produced which result. Comment each change relative to PASSING the challenge.
OVERRIDES = {
    # Enforcement phase for the 3 market-condition signals (volatility / spread / stationarity).
    # "free" (DEFAULT) = OBSERVATION-ONLY: the bot SEES the signals and LEARNS to trade both
    # stationary AND non-stationary conditions itself — no hard blocking. This is what lets the
    # policy trade enough to learn to pass (the old hard gates shut ~98.7% of opens on EURUSD).
    # "constrained" = the stationarity signal re-enforces (blocks opens when non-stationary) —
    # a LATE hardening step only. (engine.py reads config.TRAINING_PHASE.)
    "training_phase": "free",

    # Training wheels (operator counter-trend OPEN blocks on 30m+4H).
    "training_wheels": True,            # True = enforce, False = remove

    # Challenge numbers — passed to make_challenge(); YOU set these.
    "daily_target_pct": 2.5,            # your daily profit goal (% of starting balance)
    "daily_risk_pct":   4.0,            # the trailing-wall risk allowed to make it
    "permanent_dd_pct": 10.0,           # the -10% permanent max-overall-loss wall — OBSERVATION
                                        # ONLY (C12 dist_to_perm_dd scalar), NOT enforced in training.

    # ── NOTE (2026-06-18 redesign) ───────────────────────────────────────────
    # The 3 former "gates" became OBSERVATION signals: the old gate-threshold knobs
    # (adf_p_value_threshold / atr_min_multiplier / spread_max_pips) were REMOVED — there is
    # nothing to "loosen" anymore; the only enforcement knob is training_phase. Operator-tunable
    # REWARD weights are PLANNED (reward.py uses internal constants today), so they are NOT here yet.
}
print("INPUTS + OVERRIDES set. Now run Cells 4-8.")

## Cell 4 — Load or init the policy + checkpoint compatibility check

In [ ]:
# ── Cell 4 ─ Policy init + compatibility gate ──
import json, hashlib, datetime
from pathlib import Path
from quantra.learning_system.ppo_agent.agent import PPOAgent

REGISTRY_DIR = Path("artifacts/policy_registry")
STATE_DIM = RT.nominal_state_dim   # 🔴 LOCKED observation width (207, or 189 w/o raw inputs)

# Baseline the OVERRIDES diff is measured against — this drives the AUTO-NAME (Cell 6).
BASELINE = {
    "training_phase": "free", "training_wheels": True,
    "daily_target_pct": 2.5, "daily_risk_pct": 4.0, "permanent_dd_pct": 10.0,
}

class CompatibilityError(RuntimeError):
    pass

def compatibility_signature(state_dim, overrides):
    # Hash of state_dim + reward-layer SHAPE + law fingerprint. Changing STATE_DIM or the
    # reward SHAPE (not just a weight value) changes this → resume is refused.
    reward_shape = sorted(k for k in overrides if k.startswith("reward_"))  # none today (reward redesign pending)
    payload = {"state_dim": state_dim, "reward_shape": reward_shape,
               "law_fingerprint": "laws9+signals3"}  # TODO(wire): hash real law params from laws.py
    return "sha256:" + hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

CUR_SIG = compatibility_signature(STATE_DIM, OVERRIDES)
agent = PPOAgent()   # reads STATE_DIM dynamically — no hardcoded input dim

if RESUME_FROM:
    man_path = Path(RESUME_FROM).with_name("manifest.json")
    if man_path.exists():
        old = json.loads(man_path.read_text())
        if old.get("compatibility_signature") != CUR_SIG:
            # The OLD checkpoint is always preserved first; old policies are NEVER deleted.
            raise CompatibilityError(
                "Cannot resume: the checkpoint's compatibility signature does not match the "
                "current config/OVERRIDES.\n"
                f"  saved:   {old.get('compatibility_signature')}\n"
                f"  current: {CUR_SIG}\n"
                "WHY: STATE_DIM or the reward SHAPE changed, so the old network's input/output "
                "assumptions no longer hold. Start FRESH (set RESUME_FROM=None) or revert the change.")
    # TODO(wire): agent.load(RESUME_FROM) once fast-loop checkpoint I/O is connected.
    print("Resuming from", RESUME_FROM, "(compatibility OK)")
else:
    print("Starting a FRESH policy (RESUME_FROM is None).")
print("Compatibility signature:", CUR_SIG)

## Cell 5 — THE BARBERSHOP LOOP (live output · clean stop-on-interrupt)

In [ ]:
# ── Cell 5 ─ The loop. Hit Colab's STOP any time — the interrupt is caught and a clean
# checkpoint is saved before exit. You NEVER lose weights on stop.

# DEMO_MODE prints the LIVE OUTPUT FORMAT with clearly-LABELLED PLACEHOLDER metrics so you
# can see/checkpoint the loop end-to-end before the env is wired. Set False once
# barbershop_run_day() is connected to TradingEnv for REAL metrics.
DEMO_MODE = True

def barbershop_run_day(agent, start_date, day_index, overrides):
    # ── THE ONE INTEGRATION POINT TO WIRE ────────────────────────────────────
    #   1. cfg.make_challenge(daily_target_pct=overrides["daily_target_pct"],
    #      daily_risk_pct=overrides["daily_risk_pct"]) for one day + a TradingEnv on that day;
    #   2. apply OVERRIDES at construction: training_phase -> config.TRAINING_PHASE,
    #      training_wheels -> config (the 3 market signals are observation-only in "free");
    #   3. run the DETERMINISTIC policy for the day, then return the per-day scoreboard dict.
    if not DEMO_MODE:
        raise NotImplementedError("Wire barbershop_run_day() to TradingEnv (see steps above).")
    # --- PLACEHOLDER (NOT REAL) — deterministic so the plumbing is demonstrable ---
    import random; rng = random.Random(1000 * day_index + 7)
    pnl = round(rng.uniform(-2.0, 2.8), 1)
    dd  = round(-abs(rng.uniform(0.5, 4.0)), 1)
    breached = dd <= -4.0
    passed   = (pnl >= 2.5) and not breached
    trades   = rng.randint(0, 9)
    # blocked_rate = fraction of opens the signals blocked. 0 in PHASE_FREE (observation-only);
    # >0 only in PHASE_CONSTRAINED. Kept in the registry schema (avg_gate_block_rate) for continuity.
    blocked = 0.0 if overrides.get("training_phase", "free") == "free" else round(rng.uniform(0.1, 0.6), 2)
    return {"day": day_index, "passed": passed, "pnl_pct": pnl, "dd_pct": dd,
            "breached": breached, "trades": trades, "gate_block_rate": blocked}

def print_pass_table(p, n_passes, day_results):
    print(f"\nPass {p}/{n_passes}")
    for r in day_results:
        verdict = "PASS  " if r["passed"] else "FAIL  "
        dd = f"DD {r['dd_pct']:.1f}%" + (" BREACHED" if r["breached"] else "")
        print(f"  Day {r['day']}: {verdict}| {r['pnl_pct']:+.1f}% | {dd} | {r['trades']} trades")
    npass = sum(r["passed"] for r in day_results)
    avg = lambda k: sum(r[k] for r in day_results) / len(day_results)
    print(f"  Summary: {npass}/{len(day_results)} passed | Avg P&L: {avg('pnl_pct'):+.1f}% | "
          f"Avg DD: {avg('dd_pct'):.1f}% | Avg trades/day: {avg('trades'):.1f}")
    # In PHASE_FREE the bot is never blocked — a LOW trade count means the POLICY chose not to
    # trade (shape it via reward), NOT a gate lockout. (In PHASE_CONSTRAINED add a stat-blocked %.)

def summarize_pass(p, day_results):
    avg = lambda k: round(sum(r[k] for r in day_results) / len(day_results), 3)
    return {"pass": p, "days_passed": sum(r["passed"] for r in day_results),
            "days_failed": sum(not r["passed"] for r in day_results),
            "avg_pnl": avg("pnl_pct"), "avg_dd": avg("dd_pct"),
            "breach_count": sum(r["breached"] for r in day_results),
            "avg_gate_block_rate": avg("gate_block_rate")}   # 0.0 in PHASE_FREE

PASS_HISTORY = []   # consumed by Cell 6 → performance.json
if DEMO_MODE:
    print("DEMO_MODE — metrics below are LABELLED PLACEHOLDERS, not a real run.")
    print("Wire barbershop_run_day() to TradingEnv for real FTMO metrics.\n")
try:
    for p in range(1, N_PASSES + 1):
        day_results = [barbershop_run_day(agent, START_DATE, d, OVERRIDES)
                       for d in range(1, N_DAYS + 1)]
        print_pass_table(p, N_PASSES, day_results)
        PASS_HISTORY.append(summarize_pass(p, day_results))
        if p % CHECKPOINT_INTERVAL == 0:
            print(f"  [checkpoint] pass {p}: persisting weights + telemetry + registry…")
            # TODO(wire): agent.checkpoint(<drive path>) so a Colab disconnect resumes here.
except KeyboardInterrupt:
    print("\n[stop] interrupt caught — saving a clean checkpoint before exit. Weights are safe.")
    # TODO(wire): agent.checkpoint(<drive path>) + re-run Cell 6 so nothing is lost.
print("\nLoop finished. Run Cell 6 to persist telemetry + the Policy Registry entry.")

## Cell 6 — Auto-emit telemetry + write the Policy Registry entry (AUTO-NAMED)

In [ ]:
# ── Cell 6 ─ Persist telemetry + the Policy Registry entry ──
# The policy name is DERIVED from the OVERRIDES diff vs BASELINE — never hand-typed.

def auto_name(overrides, baseline, base_policy):
    # Tokens come straight from the OVERRIDES diff vs BASELINE — never hand-typed.
    changes, tokens = [], []
    # Enforcement phase (str): "free" (baseline) vs "constrained".
    if overrides.get("training_phase", "free") != baseline.get("training_phase", "free"):
        tokens.append(overrides["training_phase"]); changes.append(f"training_phase={overrides['training_phase']}")
    # Training wheels (bool).
    wheel_state = "ON" if overrides["training_wheels"] else "OFF"
    if overrides["training_wheels"] != baseline["training_wheels"]:
        tokens.append("wheels" + wheel_state.lower()); changes.append(f"training_wheels={wheel_state}")
    # Challenge numbers (float): tgt / risk / permdd.
    NUM = {"daily_target_pct": "tgt", "daily_risk_pct": "risk", "permanent_dd_pct": "permdd"}
    for k, lbl in NUM.items():
        if overrides.get(k) != baseline.get(k):
            tokens.append(f"{lbl}{overrides[k]:g}"); changes.append(f"{k}={overrides[k]:g}")
    base_v = 1
    if base_policy and base_policy.split("-")[0].startswith("v"):
        try: base_v = int(base_policy.split("-")[0][1:])
        except ValueError: base_v = 1
    version = base_v + (1 if base_policy else 0)
    name = f"v{version}-" + "-".join(tokens) if tokens else f"v{version}-baseline"
    return name, {"changes": changes, "wheel_state": wheel_state}

BASE_POLICY = None  # set to the resumed policy's name when RESUME_FROM is used
AUTO_NAME, AUTO_BASIS = auto_name(OVERRIDES, BASELINE, BASE_POLICY)
print("Auto-generated policy name:", AUTO_NAME, " (operator label was:", POLICY_NAME, ")")

pdir = REGISTRY_DIR / AUTO_NAME
pdir.mkdir(parents=True, exist_ok=True)

manifest = {
    "policy_name": AUTO_NAME,
    "auto_name_basis": AUTO_BASIS,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "base_policy": BASE_POLICY,
    "data_window": {"start": START_DATE, "n_days": N_DAYS},
    "n_passes_completed": len(PASS_HISTORY),
    "state_dim": STATE_DIM,
    "training_wheels": OVERRIDES["training_wheels"],
    "training_phase": OVERRIDES["training_phase"],
    "overrides_applied": OVERRIDES,
    "compatibility_signature": CUR_SIG,
}
(pdir / "manifest.json").write_text(json.dumps(manifest, indent=2))

best = max(PASS_HISTORY, key=lambda x: x["days_passed"]) if PASS_HISTORY else None
performance = {
    "pass_history": PASS_HISTORY,
    "best_pass": best,
    "overall_pass_rate": round(sum(x["days_passed"] for x in PASS_HISTORY) /
                               max(1, len(PASS_HISTORY) * N_DAYS), 3),
}
(pdir / "performance.json").write_text(json.dumps(performance, indent=2))
(pdir / "compatibility.sig").write_text(CUR_SIG)

# Telemetry for the Barbershop. Canonical PRODUCER on real bars: scripts/emit_real_telemetry.py.
RUN_ID = AUTO_NAME + "_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
tel_dir = Path("artifacts/telemetry"); tel_dir.mkdir(parents=True, exist_ok=True)
with (tel_dir / f"{RUN_ID}.jsonl").open("w") as f:
    for s in PASS_HISTORY:
        f.write(json.dumps(s) + "\n")
print("Registry written  ->", pdir)
print("Telemetry written ->", tel_dir / f"{RUN_ID}.jsonl")

## Cell 7 — Inline Barbershop charts (Screen 1 wall · Screen 3 day replay)

In [ ]:
# ── Cell 7 ─ Inline charts via barbershop/figures.py (the full 5 screens are in Cell 8) ──
from barbershop import figures, data as bdata

# Auto-source: REAL telemetry if a run exists under artifacts/telemetry/, else mock.
bundle = bdata.load_bundle(source=bdata.available_source())
print("Barbershop data source:", bundle.get("source", "?"))

# Screen 1 — Training Wall (honest 'demo curve' label until a real pass-rate series logs).
wall = bundle.get("training_wall", {}) or {}
figures.training_wall_figure(wall.get("iterations", []), wall.get("pass_rate", [])).show()

# Screen 3 — Day Replay candlesticks for the worst day. Exact bundle keys are defined in
# barbershop/contract.py + barbershop/data.py; we read defensively so the skeleton degrades
# gracefully if a key name differs.
days = bundle.get("days") or bundle.get("scoreboard") or []
if days:
    d0 = days[0]
    prices, trades = d0.get("prices"), d0.get("trades", [])
    if prices is not None:
        figures.candlestick_figure(prices, trades, tf="1m").show()
print("Screen 2 scoreboard (HTML cards) + Screens 4-5 + Risk Doctor → open the dashboard (Cell 8).")

## Cell 8 — Open the FULL Barbershop dashboard via an ngrok tunnel (READ-ONLY)

In [ ]:
# ── Cell 8 ─ ngrok tunnel to the Dash app (all 5 screens + Risk Doctor) ──
# The Barbershop is READ-ONLY: it NEVER changes training, rewards, the policy, or execution.
import threading
from pyngrok import ngrok
from barbershop.dashboard import make_app
from barbershop.data import available_source

PORT = 8050
# Set your token once (https://dashboard.ngrok.com): export NGROK_AUTHTOKEN=... before Colab,
# or paste it here. Without a token ngrok allows a limited free tunnel.
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

def _serve():
    make_app(source=available_source()).run(host="0.0.0.0", port=PORT, debug=False)

threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)
public_url = ngrok.connect(PORT, "http").public_url
print("Barbershop dashboard (all 5 screens + Risk Doctor):", public_url)